# resistAD — Genetics Pipeline on DNAnexus RAP

**Self-contained notebook to run on UK Biobank RAP JupyterLab.**

This notebook:
1. Installs plink2
2. Extracts APOE genotype (rs429358/rs7412) from directly-genotyped data
3. Computes AD polygenic risk score (Bellenguez 2022 weights) from imputed data
4. Saves PRS + APOE to files used by the resistAD pipeline
5. Re-runs cohort definition with proper genetic-risk-based stratification
6. Re-runs downstream analysis (proteomics, imaging, integration)

**Prerequisites:**
- Running inside DNAnexus RAP JupyterLab (Application 151418)
- Project: Precision Omics (`project-GzgP2K8J36636XYX12fyBjKB`)
- resistAD repo cloned or uploaded to the JupyterLab environment

**Data paths on RAP:**
- Imputed genotypes (BGEN): `/mnt/project/Bulk/Imputation/UKB imputation from genotype/`
- Genotype calls (PLINK): `/mnt/project/Bulk/Genotype Results/Genotype calls/`

---

## 0. Configuration

Edit these paths if your project layout differs.

In [2]:
import os, sys
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
# Set REPO_DIR to wherever you cloned/uploaded the resistAD repository
REPO_DIR = Path("/opt/notebooks/resistAD")  # adjust if different
assert REPO_DIR.exists(), f"resistAD repo not found at {REPO_DIR}. Clone it first."

DATA_DIR   = REPO_DIR / "data" / "raw"
OUT_DIR    = REPO_DIR / "analysis" / "out"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# DNAnexus project-mounted data
GENO_CALLS_DIR = Path("/mnt/project/Bulk/Genotype Results/Genotype calls")
IMPUTED_DIR    = Path("/mnt/project/Bulk/Imputation/UKB imputation from genotype")

# Verify data exists
assert GENO_CALLS_DIR.exists(), f"Genotype calls not found: {GENO_CALLS_DIR}"
assert IMPUTED_DIR.exists(), f"Imputed data not found: {IMPUTED_DIR}"

# Add repo to Python path
sys.path.insert(0, str(REPO_DIR))

print(f"Repo:           {REPO_DIR}")
print(f"Data dir:       {DATA_DIR}")
print(f"Genotype calls: {GENO_CALLS_DIR}")
print(f"Imputed dir:    {IMPUTED_DIR}")

Repo:           /opt/notebooks/resistAD
Data dir:       /opt/notebooks/resistAD/data/raw
Genotype calls: /mnt/project/Bulk/Genotype Results/Genotype calls
Imputed dir:    /mnt/project/Bulk/Imputation/UKB imputation from genotype


## 1. Install plink2

In [8]:
%%bash
# Download and install plink2 (Linux AVX2 build)
if command -v plink2 &> /dev/null; then
    echo "plink2 already installed: $(plink2 --version)"
else
    echo "Downloading plink2 ..."
    cd /tmp
    wget -q https://s3.amazonaws.com/plink2-assets/plink2_linux_avx2_20260311.zip
    unzip -o plink2_linux_avx2_20260311.zip
    chmod +x plink2
    sudo mv plink2 /usr/local/bin/
    rm -f plink2_linux_avx2_20260311.zip vcf_subset
    echo "Installed: $(plink2 --version)"
fi

SyntaxError: incomplete input (223996751.py, line 1)

In [9]:
!wget -q https://s3.amazonaws.com/plink2-assets/plink2_linux_avx2_20260311.zip -O /tmp/plink2.zip && \
 cd /tmp && unzip -o plink2.zip && chmod +x plink2 && mv plink2 /usr/local/bin/ && \
 plink2 --version

Archive:  plink2.zip
  inflating: plink2                  
  inflating: vcf_subset              
  inflating: intel-simplified-software-license.txt  
PLINK v2.0.0-a.7LM AVX2 Intel (11 Mar 2026)


## 2. Extract APOE genotype (rs429358 / rs7412)

APOE isoform is determined by two SNPs on chromosome 19:
- **rs429358** (19:44908684, C→T): C allele defines ε4
- **rs7412** (19:44908822, C→T): T allele defines ε2

We extract these from the directly-genotyped PLINK files (chr19).

In [10]:
from pathlib import Path
DATA_DIR = Path("/opt/notebooks/resistAD/data/raw")
print("APOE:", (DATA_DIR / "ukb_apoe_genotype.parquet").exists())
print("Score file:", (DATA_DIR / "prs_score_snps.tsv").exists())
print("PRS dir:", (DATA_DIR / "prs_per_chr").exists())

APOE: False
Score file: False
PRS dir: False


In [11]:
import subprocess

APOE_SNPS = ["rs429358", "rs7412"]
CHR19_BED = GENO_CALLS_DIR / "ukb22418_c19_b0_v2"

apoe_dir = DATA_DIR / "apoe_extract"
apoe_dir.mkdir(exist_ok=True)

# Write SNP list
snp_list = apoe_dir / "apoe_snps.txt"
snp_list.write_text("\n".join(APOE_SNPS))

# Extract with plink2
cmd = [
    "plink2",
    "--bfile", str(CHR19_BED),
    "--extract", str(snp_list),
    "--export", "A",          # additive allele count (0/1/2 dosage per SNP)
    "--out", str(apoe_dir / "apoe_dosage"),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
    raise RuntimeError("plink2 APOE extraction failed")
print("APOE extraction complete.")

Running: plink2 --bfile /mnt/project/Bulk/Genotype Results/Genotype calls/ukb22418_c19_b0_v2 --extract /opt/notebooks/resistAD/data/raw/apoe_extract/apoe_snps.txt --export A --out /opt/notebooks/resistAD/data/raw/apoe_extract/apoe_dosage
3839404142434445464748495051525354555657585960616263646566676869707172737475767778798081828384858687888990919293949596979899done.
--export A: /opt/notebooks/resistAD/data/raw/apoe_extract/apoe_dosage.raw
written.
End time: Tue Apr  7 09:45:09 2026

APOE extraction complete.


In [12]:
import pandas as pd
import numpy as np

# Load the .raw file (additive allele counts)
raw_path = apoe_dir / "apoe_dosage.raw"
apoe_raw = pd.read_csv(raw_path, sep="\\s+", engine="python")
print(f"APOE dosages loaded: {len(apoe_raw)} participants")
print(f"Columns: {apoe_raw.columns.tolist()}")

# Find the rs429358 and rs7412 columns (plink2 appends _<allele>)
rs429_col = [c for c in apoe_raw.columns if "rs429358" in c]
rs7412_col = [c for c in apoe_raw.columns if "rs7412" in c]

if not rs429_col or not rs7412_col:
    print("WARNING: APOE SNPs not found in genotype data!")
    print("Available columns:", apoe_raw.columns.tolist())
else:
    rs429_col = rs429_col[0]
    rs7412_col = rs7412_col[0]
    print(f"rs429358 column: {rs429_col}")
    print(f"rs7412 column:   {rs7412_col}")

    # Derive APOE genotype
    # rs429358: count of C allele (e4-defining). Dosage = 0, 1, or 2 copies.
    # rs7412:   count of T allele (e2-defining). Dosage = 0, 1, or 2 copies.
    def classify_apoe(e4_dose, e2_dose):
        """Map allele dosages to APOE isoform."""
        if pd.isna(e4_dose) or pd.isna(e2_dose):
            return "unknown"
        e4 = int(round(e4_dose))
        e2 = int(round(e2_dose))
        e3 = max(0, 2 - e4 - e2)
        alleles = sorted(["e4"] * e4 + ["e2"] * e2 + ["e3"] * e3)
        return "".join(alleles) if len(alleles) == 2 else "unknown"

    apoe_raw["apoe_genotype"] = [
        classify_apoe(e4, e2)
        for e4, e2 in zip(apoe_raw[rs429_col], apoe_raw[rs7412_col])
    ]
    apoe_raw["apoe_e4"] = apoe_raw["apoe_genotype"].str.contains("e4")

    # Summary
    print("\nAPOE genotype distribution:")
    print(apoe_raw["apoe_genotype"].value_counts())
    print(f"\ne4 carriers: {apoe_raw['apoe_e4'].sum()} ({apoe_raw['apoe_e4'].mean()*100:.1f}%)")

    # Save
    apoe_out = apoe_raw[["IID", "apoe_genotype", "apoe_e4"]].rename(columns={"IID": "eid"})
    apoe_out["eid"] = apoe_out["eid"].astype(str)
    apoe_out.to_parquet(DATA_DIR / "ukb_apoe_genotype.parquet", index=False)
    print(f"\nSaved: {DATA_DIR / 'ukb_apoe_genotype.parquet'}")

APOE dosages loaded: 488377 participants
Columns: ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENOTYPE', 'rs429358_C', 'rs7412_T']
rs429358 column: rs429358_C
rs7412 column:   rs7412_T

APOE genotype distribution:
apoe_genotype
e3e3       242715
e3e4        97598
unknown     75158
e2e3        50280
e2e4        10456
e4e4         9834
e2e2         2336
Name: count, dtype: int64

e4 carriers: 117888 (24.1%)

Saved: /opt/notebooks/resistAD/data/raw/ukb_apoe_genotype.parquet


## 3. Compute AD Polygenic Risk Score (PRS)

Uses Bellenguez et al. 2022 AD GWAS summary statistics with clump+threshold approach.

**Steps:**
1. Download GWAS summary statistics
2. Prepare score file (filter by p-value, exclude APOE region)
3. Run plink2 `--score` on each chromosome's imputed BGEN data
4. Combine per-chromosome scores

This takes ~30-60 minutes depending on RAP instance size.

In [14]:
# ── 3a. Download GWAS summary statistics ─────────────────────────────────────
# Bellenguez 2022: https://doi.org/10.1038/s41588-022-01024-z
# GWAS Catalog: GCST90027158

import subprocess
from pathlib import Path

DATA_DIR = Path("/opt/notebooks/resistAD/data/raw")
GWAS_FILE = DATA_DIR / "bellenguez2022_ad_gwas.tsv.gz"
GWAS_FILE.unlink(missing_ok=True)

urls = [
    "https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST90027001-GCST90028000/GCST90027158/harmonised/GCST90027158.h.tsv.gz",
    "https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST90027001-GCST90028000/GCST90027158/GCST90027158_buildGRCh38.tsv.gz",
    "http://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST90027001-GCST90028000/GCST90027158/harmonised/GCST90027158.h.tsv.gz",
]

for url in urls:
    print(f"Trying: {url}")
    result = subprocess.run(
        ["wget", "-q", "--timeout=60", "-O", str(GWAS_FILE), url],
        capture_output=True, text=True, timeout=120
    )
    if GWAS_FILE.exists() and GWAS_FILE.stat().st_size > 10000:
        print(f"Success! {GWAS_FILE.stat().st_size / 1e6:.1f} MB")
        break
    else:
        GWAS_FILE.unlink(missing_ok=True)
        print(f"  Failed ({GWAS_FILE.stat().st_size if GWAS_FILE.exists() else 0} bytes)")

if not GWAS_FILE.exists() or GWAS_FILE.stat().st_size < 10000:
    print("\nAll URLs failed. Try downloading manually:")
    print("  1. On your Mac: wget the file from https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST90027001-GCST90028000/GCST90027158/harmonised/GCST90027158.h.tsv.gz")
    print("  2. Upload to RAP: dx upload GCST90027158.h.tsv.gz --path /opt/notebooks/resistAD/data/raw/bellenguez2022_ad_gwas.tsv.gz")


Trying: https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST90027001-GCST90028000/GCST90027158/harmonised/GCST90027158.h.tsv.gz
  Failed (0 bytes)
Trying: https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST90027001-GCST90028000/GCST90027158/GCST90027158_buildGRCh38.tsv.gz
Success! 755.2 MB


In [16]:
# ── 3b. Prepare PRS score file ───────────────────────────────────────────────

P_THRESHOLD = 5e-8       # genome-wide significance
EXCLUDE_APOE = True      # exclude APOE region to separate APOE from polygenic risk
APOE_CHR = "19"
APOE_START_BP = 44_400_000
APOE_END_BP   = 46_500_000

gwas = pd.read_csv(GWAS_FILE, sep="\t", low_memory=False)
print(f"GWAS loaded: {len(gwas)} variants")
print(f"Columns: {gwas.columns.tolist()}")

# Identify required columns (varies by GWAS catalog format)
# Common formats: variant_id/SNP, chromosome/CHR, base_pair_location/BP,
#                 effect_allele/A1, beta/BETA/OR, p_value/P
col_map = {}
for cand, target in [
    (["variant_id", "SNP", "rsid", "hm_rsid"], "SNP"),
    (["chromosome", "CHR", "hm_chrom", "chr_name"], "CHR"),
    (["base_pair_location", "BP", "hm_pos", "chr_position"], "BP"),
    (["effect_allele", "A1", "hm_effect_allele"], "A1"),
    (["other_allele", "A2", "hm_other_allele"], "A2"),
    (["beta", "BETA", "hm_beta"], "BETA"),
    (["p_value", "P", "pvalue", "hm_pvalue"], "P"),
]:
    for c in cand:
        if c in gwas.columns:
            col_map[target] = c
            break

print(f"Column mapping: {col_map}")
missing = {"SNP", "CHR", "BP", "A1", "BETA", "P"} - set(col_map.keys())
if missing:
    raise ValueError(f"Missing required GWAS columns: {missing}. Available: {gwas.columns.tolist()}")

# Standardise columns
score = gwas.rename(columns={v: k for k, v in col_map.items()}).copy()
score["CHR"] = score["CHR"].astype(str).str.replace("chr", "")
score["P"] = pd.to_numeric(score["P"], errors="coerce")
score["BETA"] = pd.to_numeric(score["BETA"], errors="coerce")
score["BP"] = pd.to_numeric(score["BP"], errors="coerce")

# If BETA column has odds ratios instead of log-OR, convert
if score["BETA"].median() > 0.5:  # likely OR scale
    print("Detected OR scale — converting to log-OR")
    score["BETA"] = np.log(score["BETA"])

# Filter by p-value
score = score[score["P"] < P_THRESHOLD].copy()
print(f"After p < {P_THRESHOLD}: {len(score)} variants")

# Exclude APOE region
if EXCLUDE_APOE:
    apoe_mask = (
        (score["CHR"] == APOE_CHR) &
        (score["BP"] >= APOE_START_BP) &
        (score["BP"] <= APOE_END_BP)
    )
    n_apoe = apoe_mask.sum()
    score = score[~apoe_mask]
    print(f"Excluded {n_apoe} variants in APOE region (chr19:{APOE_START_BP}-{APOE_END_BP})")

# Drop duplicates by SNP
score = score.drop_duplicates(subset="SNP", keep="first")

# Write score file for plink2
score_file = DATA_DIR / "prs_score_snps.tsv"
score[["SNP", "A1", "BETA"]].to_csv(score_file, sep="\t", index=False, header=False)
print(f"\nScore file: {score_file} ({len(score)} SNPs)")
print(score[["SNP", "CHR", "BP", "A1", "BETA", "P"]].head(10))

GWAS loaded: 21101114 variants
Columns: ['variant_id', 'p_value', 'chromosome', 'base_pair_location', 'effect_allele', 'other_allele', 'effect_allele_frequency', 'odds_ratio', 'ci_lower', 'ci_upper', 'beta', 'standard_error', 'n_cases', 'n_controls', 'het_isq', 'het_pvalue', 'variant_alternate_id']
Column mapping: {'SNP': 'variant_id', 'CHR': 'chromosome', 'BP': 'base_pair_location', 'A1': 'effect_allele', 'A2': 'other_allele', 'BETA': 'beta', 'P': 'p_value'}
After p < 5e-08: 5637 variants
Excluded 917 variants in APOE region (chr19:44400000-46500000)

Score file: /opt/notebooks/resistAD/data/raw/prs_score_snps.tsv (4720 SNPs)
               SNP CHR         BP A1    BETA             P
991505  rs10797093   1  161133655  T -0.0491  2.677000e-08
991522  rs11265557   1  161136564  T -0.0494  2.242000e-08
991541  rs12041364   1  161140557  A  0.0494  2.206000e-08
991551  rs11591206   1  161141573  T  0.0493  2.326000e-08
991719  rs12741203   1  161170161  T  0.0492  2.457000e-08
991741  rs1

In [ ]:
# ── 3c. Run plink2 --score on each chromosome ───────────────────────────────

import glob

bgen_files = sorted(glob.glob(str(IMPUTED_DIR / "ukb22828_c*_b0_v3.bgen")))
sample_file = sorted(glob.glob(str(IMPUTED_DIR / "ukb22828_c*_b0_v3.sample")))[0]

print(f"Found {len(bgen_files)} chromosomes")
print(f"Sample file: {sample_file}")

prs_dir = DATA_DIR / "prs_per_chr"
prs_dir.mkdir(exist_ok=True)

score_file = str(DATA_DIR / "prs_score_snps.tsv")

for bgen in bgen_files:
    chrom = Path(bgen).stem.split("_c")[1].split("_")[0]
    out_prefix = str(prs_dir / f"prs_chr{chrom}")
    
    if Path(f"{out_prefix}.sscore").exists():
        print(f"  chr{chrom}: already computed, skipping")
        continue
    
    print(f"  chr{chrom}: scoring ...")
    cmd = [
        "plink2",
        "--bgen", bgen, "ref-first",
        "--sample", sample_file,
        "--rm-dup", "force-first",
        "--score", score_file,
        "--out", out_prefix,
        "--threads", "4",
        "--memory", "8000",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        if "0 variants" in result.stderr or "no valid" in result.stderr.lower():
            print(f"  chr{chrom}: no matching SNPs")
        else:
            print(f"  chr{chrom}: FAILED")
            print(result.stderr[-200:])
    else:
        print(f"  chr{chrom}: done")

print("\nPer-chromosome scoring complete.")

Found 24 chromosomes
Sample file: /mnt/project/Bulk/Imputation/UKB imputation from genotype/ukb22828_c10_b0_v3.sample
  chr10: scoring ...


In [ ]:
# ── 3d. Combine per-chromosome PRS into a single score ───────────────────────

sscore_files = sorted(glob.glob(str(prs_dir / "prs_chr*.sscore")))
print(f"Found {len(sscore_files)} per-chromosome score files")

combined = None
for sf in sscore_files:
    df = pd.read_csv(sf, sep="\t")
    # plink2 .sscore has: #IID, ALLELE_CT, NAMED_ALLELE_DOSAGE_SUM, SCORE1_AVG or SCORE1_SUM
    score_col = [c for c in df.columns if "SCORE" in c and "SUM" in c]
    if not score_col:
        score_col = [c for c in df.columns if "SCORE" in c]
    if not score_col:
        print(f"  Skipping {sf}: no SCORE column found. Cols: {df.columns.tolist()}")
        continue
    score_col = score_col[0]
    
    # Use #IID or IID
    id_col = "#IID" if "#IID" in df.columns else "IID"
    
    chunk = df[[id_col, score_col]].rename(columns={id_col: "eid", score_col: "prs_chr"})
    
    if combined is None:
        combined = chunk.rename(columns={"prs_chr": "prs_score"})
    else:
        combined = combined.merge(chunk, on="eid", how="outer")
        combined["prs_score"] = combined["prs_score"].fillna(0) + combined["prs_chr"].fillna(0)
        combined = combined.drop(columns=["prs_chr"])

# Standardise
combined["eid"] = combined["eid"].astype(str)
combined["prs_score"] = (
    (combined["prs_score"] - combined["prs_score"].mean()) / combined["prs_score"].std()
)

print(f"\nCombined PRS: {len(combined)} participants")
print(f"Mean: {combined['prs_score'].mean():.4f}, SD: {combined['prs_score'].std():.4f}")
print(combined["prs_score"].describe())

# Save
prs_out = DATA_DIR / "prs_scores.parquet"
combined.to_parquet(prs_out, index=False)
print(f"\nSaved: {prs_out}")

## 4. Merge APOE into phenotypes

Add APOE genotype columns to the existing phenotype file so `assign_cohort()` can use them.

In [ ]:
# Load existing phenotypes
pheno = pd.read_parquet(DATA_DIR / "ukb_phenotypes.parquet")
pheno["eid"] = pheno["eid"].astype(str)
print(f"Phenotypes: {len(pheno)} participants")

# Load APOE
apoe = pd.read_parquet(DATA_DIR / "ukb_apoe_genotype.parquet")
apoe["eid"] = apoe["eid"].astype(str)
print(f"APOE data: {len(apoe)} participants")

# Drop old APOE columns if they exist, then merge
for col in ["apoe_genotype", "apoe_e4", "apoe_rs429358", "apoe_rs7412"]:
    if col in pheno.columns:
        pheno = pheno.drop(columns=[col])

pheno = pheno.merge(apoe[["eid", "apoe_genotype", "apoe_e4"]], on="eid", how="left")

# Also add rs429358/rs7412 columns for extract_apoe_genotype() compatibility
# (The cohort module checks for these columns)
pheno["apoe_rs429358"] = "present"  # marker that APOE was extracted externally
pheno["apoe_rs7412"] = "present"

# Save updated phenotypes
pheno.to_parquet(DATA_DIR / "ukb_phenotypes.parquet", index=False)

print(f"\nUpdated phenotypes saved with APOE columns.")
print(f"e4 carriers: {pheno['apoe_e4'].sum()} ({pheno['apoe_e4'].mean()*100:.1f}%)")
print(f"\nGenotype distribution:")
print(pheno["apoe_genotype"].value_counts())

## 5. Re-run cohort definition with real PRS + APOE

Now `assign_cohort()` will use:
- **APOE e4 carrier** status (from genotype extraction above)
- **AD-PRS** top 10% decile (from Bellenguez 2022 scoring above)
- **HES dementia** diagnoses (from step 01)
- **Fluid intelligence** age-adjusted (from step 01)

Cohort groups:
- **Resilient** = high genetic risk (APOE e4 OR top-10% PRS) + cognitively intact + age >= 60
- **Vulnerable** = high genetic risk + dementia diagnosis
- **Control** = low genetic risk + cognitively intact

In [ ]:
%%bash
cd $HOME/resistAD  # adjust if different
python scripts/02_define_cohorts.py --prs data/raw/prs_scores.parquet

In [ ]:
# Quick check of new cohort
cohort = pd.read_parquet(OUT_DIR / "cohorts.parquet")
print("Cohort distribution:")
print(cohort["cohort"].value_counts())
print(f"\nTotal: {len(cohort)} participants")
print(f"\nAPOE e4 carrier rate by group:")
print(cohort.groupby("cohort")["apoe_e4"].mean().map(lambda x: f"{x*100:.1f}%"))

## 6. Re-run step 03 (genetics analysis with real APOE + PRS)

In [ ]:
%%bash
cd $HOME/resistAD  # adjust if different
python scripts/03_genetics.py --skip-prs --skip-rare
# PRS is already computed above; APOE analysis will now work with real genotypes.
# WES rare-variant analysis can be run separately if needed.

## 7. Re-run downstream pipeline (steps 04–07)

With properly stratified cohorts, we expect stronger proteomic and imaging signals.

In [ ]:
%%bash
cd $HOME/resistAD  # adjust if different

echo "========== Step 04: Proteomics =========="
python scripts/04_proteomics.py

echo ""
echo "========== Step 05: Imaging =========="
python scripts/05_imaging.py

echo ""
echo "========== Step 06: Methylation =========="
python scripts/06_methylation.py

echo ""
echo "========== Step 07: Integration =========="
python scripts/07_integration.py --skip-mofa

echo ""
echo "Pipeline complete!"

## 8. Summary of results

In [ ]:
print("=" * 60)
print("resistAD Pipeline Results Summary")
print("=" * 60)

# Cohort
cohort = pd.read_parquet(OUT_DIR / "cohorts.parquet")
print("\n-- Cohort --")
for g, n in cohort["cohort"].value_counts().items():
    e4_pct = cohort[cohort["cohort"] == g]["apoe_e4"].mean() * 100
    print(f"  {g:<12}: {n:>6}  (APOE e4: {e4_pct:.1f}%)")

# Proteomics
prot_path = OUT_DIR / "proteomics" / "protein_de_resilient_vs_vulnerable.csv"
if prot_path.exists():
    prot = pd.read_csv(prot_path)
    n_sig = (prot["padj"] < 0.05).sum() if not prot.empty and "padj" in prot.columns else 0
    print(f"\n-- Proteomics --")
    print(f"  Proteins tested: {len(prot)}")
    print(f"  Significant (FDR<0.05): {n_sig}")
    if n_sig > 0:
        print("  Top hits:")
        print(prot[prot["padj"] < 0.05][["protein", "log2FC", "padj"]].head(10).to_string(index=False))

# Imaging
idp_path = OUT_DIR / "imaging" / "idp_differences.csv"
if idp_path.exists():
    idp = pd.read_csv(idp_path)
    print(f"\n-- Imaging --")
    print(f"  IDPs tested: {len(idp)}")
    if "padj_Resilient" in idp.columns:
        sig_r = (idp["padj_Resilient"] < 0.05).sum()
        sig_v = (idp["padj_Vulnerable"] < 0.05).sum()
        print(f"  Significant (Resilient vs Control): {sig_r}")
        print(f"  Significant (Vulnerable vs Control): {sig_v}")

# Integration
sig_path = OUT_DIR / "integration" / "resilience_signature.csv"
if sig_path.exists():
    sig = pd.read_csv(sig_path)
    print(f"\n-- Integration --")
    print(f"  Resilience signature: {len(sig)} features")

print("\n" + "=" * 60)
print(f"All results saved to: {OUT_DIR}")

---

## Next steps

After running this notebook:

1. **Download results** (non-participant-level only): aggregate statistics, plots, and signatures can be exported
2. **WES rare-variant analysis**: run `python scripts/03_genetics.py --bfile <WES_PLINK_PREFIX>` with the exome PLINK files
3. **MOFA+ integration**: remove `--skip-mofa` from step 07 if muon/mofapy2 are installed
4. **Iterate**: re-run cohort definition with adjusted thresholds as needed